# Track 1 model preparation
CPU-only: verify the approved private checkpoint files and fetch/cache public models. No reconstruction or scoring claim.

In [ ]:
import os,sys,json,subprocess,hashlib,tarfile,shutil
from pathlib import Path
py=sys.executable;env=os.environ.copy()
def file_sha256(path):
 digest=hashlib.sha256()
 with path.open('rb') as handle:
  for chunk in iter(lambda:handle.read(16*1024*1024),b''):digest.update(chunk)
 return digest.hexdigest()
subprocess.run([py,'-m','pip','install','gdown','huggingface_hub'],check=True)
repo=Path('/tmp/video_to_data');env['GIT_LFS_SKIP_SMUDGE']='1'
subprocess.run(['git','clone','--filter=blob:none','--no-checkout','https://github.com/nvidia-isaac/video_to_data.git',str(repo)],env=env,check=True)
subprocess.run(['git','sparse-checkout','set','reconstruction/modules/v2d_foundation_pose','reconstruction/modules/v2d_cari4d'],cwd=repo,env=env,check=True)
subprocess.run(['git','checkout','8c778f905cb35f76f8e6cd6e2ea486da4fb9e0ef'],cwd=repo,env=env,check=True)
modules=repo/'reconstruction/modules'
subprocess.run([py,'-m','pip','install','--no-deps','-e',str(modules/'v2d_foundation_pose/lib'),'-e',str(modules/'v2d_cari4d/lib')],env=env,check=True)


In [ ]:
weights=Path('/tmp/track1_weights');weights.mkdir(exist_ok=True)
manifests=[json.loads(p.read_text()) for p in Path('/kaggle/input').rglob('model_manifest.json')]
manifest=next(value for value in manifests if isinstance(value,list) and len(value)==8 and all('file' in x and 'sha256' in x for x in value))
archives=list(Path('/kaggle/input').rglob('track1_models.tar'))
if len(archives)==1:
 with tarfile.open(archives[0]) as archive:archive.extractall(weights,filter='data')
else:
 # Kaggle expands uploaded .tar files into a directory named after the archive.
 roots=[p.parents[2] for p in Path('/kaggle/input').rglob('step200000.pth') if p.parent.name=='2026-08-25-09-35-57']
 assert len(roots)==1,roots
 for item in manifest:
  source=roots[0]/item['file'];destination=weights/item['file']
  destination.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(source,destination)
for item in manifest:
 path=weights/item['file'];assert path.stat().st_size==item['bytes']
 assert file_sha256(path)==item['sha256'],path
# T4 lacks bfloat16. Use full precision for the human initializer to avoid FP16 overflow.
config=weights/'sam3d_body/checkpoints/sam-3d-body-dinov3/model_config.yaml'
original_config=config.read_text()
assert 'USE_FP16: true' in original_config
config.write_text(original_config.replace('USE_FP16: true','USE_FP16: false'))
Path('/kaggle/working/t4_precision_manifest.json').write_text(json.dumps({
 'sam3d_precision':'float32','original_config_sha256':hashlib.sha256(original_config.encode()).hexdigest(),
 'effective_config_sha256':hashlib.sha256(config.read_bytes()).hexdigest()},indent=2))
env['HF_HOME']=str(weights/'hf_home')
public_models="""
from pathlib import Path
from v2d.cari4d.lib.download_weights import _download_moge2,_download_torch_hub_models
from v2d.foundation_pose.lib.download_weights import download_weights
from v2d.foundation_pose.lib.backends import NVLABS_PYTORCH
weights=Path('/tmp/track1_weights')
_download_moge2(weights)
_download_torch_hub_models(weights)
download_weights(str(weights/'foundationpose'),backend=NVLABS_PYTORCH)
"""
subprocess.run([py,'-c',public_models],env=env,check=True)


In [ ]:
out=Path('/kaggle/working/public_models.bin')
parts=[weights/'hf_home',weights/'sam3d_body/torch_home',weights/'foundationpose']
with tarfile.open(out,'w:gz',compresslevel=1) as archive:
 for part in parts:archive.add(part,arcname=str(part.relative_to(weights)))
Path('/kaggle/working/public_model_cache_manifest.json').write_text(json.dumps({
 'sha256':file_sha256(out),'bytes':out.stat().st_size,
 'paths':[str(p.relative_to(weights)) for p in parts],
 'contains_gated_models':False,'contains_credentials':False},indent=2))
print('Public model preparation complete',out.stat().st_size)
